# 02 — Extraction des vecteurs (étapes 1d et 1e)

Fait passer chaque image (recadrée à 128 px) dans les deux backbones gelés, en
112 et 224 px, et écrit les vecteurs sur Drive :
`MyDrive/srd-facegeom/features/<backbone>/<résolution>/<split>.npy`.

- **Serveur GPU T4 obligatoire.**
- Si Colab se coupe : relancer le notebook, l'extraction reprend au dernier
  morceau terminé.
- **Étape 1d (essai)** : `LIMIT = 1000` → 1 000 images par partie, dans un
  dossier à part `features_essai1000`. Sert à vérifier et à mesurer le temps.
- **Étape 1e (complète)** : `LIMIT = None`.

## 1. Drive et code

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/Arsenicophe/srd-facegeom.git"
CODE_DIR = Path("/content/srd-facegeom")
if CODE_DIR.exists():
    subprocess.run(["git", "-C", str(CODE_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(CODE_DIR)], check=True)
os.chdir(CODE_DIR)
sys.path.insert(0, str(CODE_DIR / "src"))
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

import torch
assert torch.cuda.is_available(), "Pas de GPU : choisir un serveur Colab avec GPU T4"
print("GPU :", torch.cuda.get_device_name(0))

## 2. Images sur le disque local de Colab (~2 min la première fois)

In [ ]:
from facegeom.data.images import ensure_local_images

IMG_DIR = ensure_local_images(
    "/content/drive/MyDrive/srd-facegeom/data/celeba/img_align_celeba.zip", "/content/data"
)
print(IMG_DIR, len(os.listdir(IMG_DIR)), "images")

## 3. Extraction

In [ ]:
LIMIT = 1000   # étape 1d : essai. Mettre None pour l'extraction complète (étape 1e).

limit_arg = f"--limit {LIMIT}" if LIMIT else ""
!PYTHONPATH=src python scripts/05_extract_features.py --source {IMG_DIR} {limit_arg}

## 4. Contrôle

In [ ]:
import json
import numpy as np

root = Path("/content/drive/MyDrive/srd-facegeom") / ("features" + (f"_essai{LIMIT}" if LIMIT else ""))
for meta_path in sorted(root.glob("*/*/meta.json")):
    d = meta_path.parent
    for split in ["train", "val", "test"]:
        x = np.load(d / f"{split}.npy", mmap_mode="r")
        n = len((d / f"{split}_images.txt").read_text().split())
        print(f"{d.parent.name:16} {d.name:>3} px {split:5} : {x.shape}, {x.dtype}, images {n}, "
              f"fini {bool(np.isfinite(x[:1000]).all())}")
runs = json.loads(meta_path.read_text())["runs"]
print(len(runs), "passage(s) ; temps du premier :")
print(json.dumps(runs[0]["timings"], indent=1))